In [2]:
import numpy as np
import netCDF4 as nc
from tqdm.auto import tqdm

def generate_helmholtz_dataset_batched(
    filepath="Helmholtz-Sinusoidal.nc", 
    n_samples=10000, 
    nx=128, 
    ny=128, 
    n_max=5, 
    batch_size=500
):
    print(f"Generating {n_samples} samples in batches of {batch_size} (FP64)...")

    # 1. Coordinate Setup
    x_vals = np.linspace(0.0, 1.0, nx, dtype=np.float64)
    y_vals = np.linspace(0.0, 1.0, ny, dtype=np.float64)
    X, Y = np.meshgrid(x_vals, y_vals, indexing='ij')

    # 2. Pre-allocate NetCDF4 File on Disk
    with nc.Dataset(filepath, 'w', format='NETCDF4') as rootgrp:
        # Define Dimensions
        rootgrp.createDimension('sample', n_samples)
        rootgrp.createDimension('x', nx)
        rootgrp.createDimension('y', ny)

        # Create Coordinate Variables
        x_var = rootgrp.createVariable('x', 'f8', ('x',))
        y_var = rootgrp.createVariable('y', 'f8', ('y',))
        x_var[:] = x_vals
        y_var[:] = y_vals

        # Create Data Variables (f8 = float64) with compression
        sol_var = rootgrp.createVariable('solution', 'f8', ('sample', 'x', 'y'), zlib=True)
        src_var = rootgrp.createVariable('source', 'f8', ('sample', 'x', 'y'), zlib=True)
        
        # NEW: Create a 1D variable to store the wave number 'k' for each sample
        k_var = rootgrp.createVariable('k', 'f8', ('sample',), zlib=True)

        # Dataset Metadata Attributes
        rootgrp.description = "2D Helmholtz Equation Dataset with Sinusoidal Superpositions"
        rootgrp.equation = "(nabla^2 + k)u = q"
        rootgrp.n_max_terms = n_max
        rootgrp.boundary_condition = "Homogeneous Dirichlet (u=0)"

        # 3. Batched Generation Loop with Progress Bar
        num_batches = int(np.ceil(n_samples / batch_size))
        
        for b in tqdm(range(num_batches), desc="Writing Dataset Batches"):
            start_idx = b * batch_size
            end_idx = min(start_idx + batch_size, n_samples)
            b_size = end_idx - start_idx

            # Randomize Parameters for current batch only
            A = np.random.uniform(-5.0, 5.0, size=(b_size, n_max)).astype(np.float64)
            num_terms = np.random.randint(1, n_max + 1, size=b_size)
            mask = np.arange(n_max) >= num_terms[:, None]
            A[mask] = 0.0

            f1 = np.random.randint(1, 6, size=(b_size, n_max)).astype(np.float64)
            f2 = np.random.randint(1, 6, size=(b_size, n_max)).astype(np.float64)
            k = np.random.uniform(1.0, 10.0, size=(b_size, 1)).astype(np.float64)

            # Reshape for broadcasting inside the batch: (b_size, n_max, 1, 1)
            A_b = A[:, :, None, None]
            f1_b = f1[:, :, None, None]
            f2_b = f2[:, :, None, None]
            k_b = k[:, :, None, None]

            X_b = X[None, None, :, :]
            Y_b = Y[None, None, :, :]

            # Compute wave components for the current batch
            sin_x = np.sin(f1_b * np.pi * X_b)
            sin_y = np.sin(f2_b * np.pi * Y_b)
            wave_components = A_b * sin_x * sin_y

            # Solution (U) and Source (Q)
            U_batch = np.sum(wave_components, axis=1)

            laplacian_multiplier = k_b - (f1_b * np.pi)**2 - (f2_b * np.pi)**2
            Q_batch = np.sum(laplacian_multiplier * wave_components, axis=1)

            # 4. Stream slice directly into the pre-allocated disk file
            sol_var[start_idx:end_idx, :, :] = U_batch
            src_var[start_idx:end_idx, :, :] = Q_batch
            
            # NEW: Stream the k values (flattened to 1D) into the file
            k_var[start_idx:end_idx] = k.flatten()

    print(f"\nSuccessfully generated and saved FP64 dataset to: {filepath}")

if __name__ == "__main__":
    generate_helmholtz_dataset_batched()

Generating 10000 samples in batches of 500 (FP64)...


Writing Dataset Batches:   0%|          | 0/20 [00:00<?, ?it/s]


Successfully generated and saved FP64 dataset to: Helmholtz-Sinusoidal.nc
